<a href="https://colab.research.google.com/github/emilsar/NLP-TCGA/blob/main/notebooks/Fall2026/lab5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5 — Better Numbers from the Same Words

Lab 4 trained logistic regression on the simplest numbers there are: 1 if a report contains a
word, 0 if not. This lab keeps the model and changes the numbers. You will weight words by how
rare they are (TF–IDF), learn why one validation pile of hospitals is not enough and use five
instead, keep only the words that tell the cancer types apart, and squeeze the columns down to a
few hundred. Then you will open the test pile, once.

As in Labs 3 and 4, every pile is **split by hospital**: every score is about hospitals the model
has never seen.

Two kinds of puzzle, as before:

- **Code puzzles**: fill in each `___` and run the cell. An `assert` line tells you
  immediately whether you got it right.
- **THINK puzzles**: no code. Write your answer on the `# ANSWER:` line, in a sentence or two.
  The book section named in the puzzle has what you need.

Some cells fit a model ten times over and take a few minutes; the puzzle says so.

The book: [Chapter 5, Feature Engineering for Text](https://obooks.tech/nlp/feature-engineering.html).

## Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score

BASE = "https://raw.githubusercontent.com/emilsar/NLP-TCGA/main/data"

# Lab 1 in four lines, plus each report's site code, exactly as in Labs 3 and 4.
reports = pd.read_csv(f"{BASE}/corpus/TCGA_Reports.csv")
labels = pd.read_csv(f"{BASE}/cancer_type/tcga_patient_to_cancer_type.csv")
reports["patient_id"] = reports["patient_filename"].apply(lambda x: x.split(".")[0])
reports["cancer_type"] = labels.set_index("patient_id").loc[reports["patient_id"], "cancer_type"].values
reports["site"] = reports["patient_id"].apply(lambda b: b.split("-")[1])

# Lab 4's three piles, split by hospital: no site appears in more than one pile.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
rows_a, rows_b = next(splitter.split(reports, groups=reports["site"]))
train_all, test = reports.iloc[rows_a], reports.iloc[rows_b]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
rows_a, rows_b = next(splitter.split(train_all, groups=train_all["site"]))
train, val = train_all.iloc[rows_a], train_all.iloc[rows_b]

print(len(train), "train,", len(val), "validation,", len(test), "test reports")
print(len(train_all), "train + validation reports, from", train_all["site"].nunique(), "hospitals")

## How much is a word worth?

A word that every report contains cannot tell reports apart. TF–IDF gives each word a weight,
its **idf**, that is larger the fewer reports contain it: $\log(n / \text{df})$, where $n$ is the
number of reports and df the number that contain the word. First, by hand, on the six toy reports
of Chapter 1.

In [ ]:
# PUZZLE 1 — the idf of every word: the log of (number of reports / reports containing the word).
# Chapter 5, "The weight of a word".

texts = [
    "invasive ductal carcinoma of the breast estrogen receptor positive",
    "breast tissue shows invasive carcinoma with ductal features",
    "clear cell renal carcinoma involving the kidney",
    "renal mass consistent with clear cell carcinoma",
    "lung adenocarcinoma with malignant gland formation",
    "pulmonary adenocarcinoma involving lung tissue",
]
toy_labels = ["Breast", "Breast", "Kidney", "Kidney", "Lung", "Lung"]

counts = CountVectorizer()
X = counts.fit_transform(texts)
in_reports = (X > 0).sum(axis=0).A1          # how many reports contain each word

idf = pd.Series(np.log(___ / in_reports), index=counts.get_feature_names_out())
print(idf.sort_values().round(2).head(8))

assert round(idf["carcinoma"], 2) == 0.41 and idf["the"] == idf["renal"]

In [ ]:
# PUZZLE 2 — THINK
# "the" and "renal" get exactly the same idf. Why? What does idf measure about a word, and what
# does it not measure?
# Chapter 5, "The weight of a word".
#
# ANSWER: ___

## TF–IDF on the real reports

Lab 4's model used word presence. Here it is against TF–IDF, each fitted on the training pile and
scored on the validation pile. TF–IDF scales every report's row to length 1, so its numbers are
small and the penalty has to be much weaker: a much larger `C`.

In [ ]:
# PUZZLE 3 — two blanks: log counts instead of raw counts (the setting that replaces a count c by
# 1 + log c), and the penalty strength 100.
# Chapter 5, "What scikit-learn computes" and "TF–IDF on the real corpus, and searching settings".
# This cell takes about a minute.

def val_score(model):
    model.fit(train["text"], train["cancer_type"])
    return (model.predict(val["text"]) == val["cancer_type"]).mean()

presence = make_pipeline(CountVectorizer(binary=True, stop_words="english"),
                         LogisticRegression(C=1, max_iter=3000))
tfidf = make_pipeline(TfidfVectorizer(sublinear_tf=___, stop_words="english"),
                      LogisticRegression(C=___, max_iter=3000))

presence_val, tfidf_val = val_score(presence), val_score(tfidf)
print("validation accuracy, presence:", round(presence_val, 3))
print("validation accuracy, TF-IDF:  ", round(tfidf_val, 3))

assert round(presence_val, 3) == 0.806 and round(tfidf_val, 3) == 0.789

In [ ]:
# PUZZLE 4 — THINK
# On the validation pile, presence wins by almost two points. Can you conclude that TF-IDF is
# worse for new hospitals? What would you want to know first?
# Chapter 5, "One validation pile is not enough".
#
# ANSWER: ___

## Five folds of hospitals

Cross-validation by hospital deals the 548 training and validation hospitals into five folds and
lets each fold take a turn as the validation pile. `cross_val_score` does the fitting and scoring,
and because the vectorizer is inside the pipeline, it is refitted in every fold.

In [ ]:
# PUZZLE 5 — the column that says which hospital each report came from (the same blank, twice).
# Chapter 5, "Cross-validation by hospital".
# This cell fits ten models and takes a few minutes.

folds = GroupKFold(n_splits=5)
presence_scores = cross_val_score(presence, train_all["text"], train_all["cancer_type"],
                                  groups=train_all[___], cv=folds)
tfidf_scores = cross_val_score(tfidf, train_all["text"], train_all["cancer_type"],
                               groups=train_all[___], cv=folds)

print(pd.DataFrame({"presence": presence_scores, "TF-IDF": tfidf_scores},
                   index=[f"fold {i}" for i in range(1, 6)]).round(3))
print("averages:", round(presence_scores.mean(), 3), round(tfidf_scores.mean(), 3))

assert round(presence_scores.mean(), 3) == 0.763 and round(tfidf_scores.mean(), 3) == 0.794

In [ ]:
# PUZZLE 6 — on how many folds does TF-IDF beat presence? `tfidf_scores > presence_scores` is an
# array of True and False; True counts as 1.
# Chapter 5, "Cross-validation by hospital" (comparing fold by fold).

wins = (tfidf_scores > presence_scores).___()

assert wins == 4
print("TF-IDF is better on", wins, "of the 5 folds")

In [ ]:
# PUZZLE 7 — THINK
# Look down one column of the table in Puzzle 5: the same setting scores very differently on
# different folds. What makes one fold harder than another? And why is it still fair to compare
# the two settings fold by fold?
# Chapter 5, "Cross-validation by hospital".
#
# ANSWER: ___

## Words that tell the types apart

Chi-square scores each word by how unevenly it is spread over the cancer types, compared with how
it would be spread if it had nothing to do with the type. First by hand, for one word of the toy
reports.

In [ ]:
# PUZZLE 8 — the chi-square of "renal": add up, over the types, the squared gap between observed
# and expected, divided by expected.
# Chapter 5, "By what a word says about the type: chi-square".

presence_vec = CountVectorizer(binary=True)
P = presence_vec.fit_transform(texts)
column = pd.Series(P[:, list(presence_vec.get_feature_names_out()).index("renal")].toarray().ravel())
y = pd.Series(toy_labels)

observed = column.groupby(y).sum()
expected = column.sum() * y.value_counts(normalize=True).sort_index()
chi_square = ((observed - expected) ** ___ / expected).sum()

print("observed:", observed.tolist(), " expected:", expected.round(2).tolist())
assert round(chi_square, 2) == 4.0
print("chi-square of 'renal':", round(chi_square, 2))

In [ ]:
# PUZZLE 9 — keep the 300 words with the highest chi-square, inside the pipeline.
# Chapter 5, "By what a word says about the type: chi-square".
# This cell takes a few minutes.

chosen = make_pipeline(TfidfVectorizer(sublinear_tf=True, stop_words="english"),
                       SelectKBest(chi2, k=___),
                       LogisticRegression(C=100, max_iter=3000))
chosen_scores = cross_val_score(chosen, train_all["text"], train_all["cancer_type"],
                                groups=train_all["site"], cv=folds)

print(pd.DataFrame({"TF-IDF, all words": tfidf_scores, "TF-IDF, 300 words": chosen_scores},
                   index=[f"fold {i}" for i in range(1, 6)]).round(3))

assert round(chosen_scores.mean(), 3) == 0.908

In [ ]:
# PUZZLE 10 — the 15 words with the highest chi-square, highest first, chosen from all the
# training and validation reports. The blank picks the last 15 positions of the sorted order.
# Appendix B, "The top-N idiom".

vec = TfidfVectorizer(sublinear_tf=True, stop_words="english")
selector = SelectKBest(chi2, k=300).fit(vec.fit_transform(train_all["text"]), train_all["cancer_type"])
words = vec.get_feature_names_out()

top = selector.scores_.argsort()[___][::-1]

assert words[top[0]] == "testis"
print(list(words[top]))

In [ ]:
# PUZZLE 11 — THINK
# Two questions. (a) Chi-square uses the labels. Why must SelectKBest sit inside the pipeline,
# rather than being fitted once on all of train_all before cross-validation? (b) The words it
# keeps are organs, tumours and operations. Why would throwing away 37,000 other words help most
# on hospitals the model has never seen?
# Chapter 5, "By what a word says about the type: chi-square".
#
# ANSWER: ___

## Squeezing the columns

Truncated SVD builds a few new columns, each a mix of all the words, keeping as much of the
reports' rows as it can. Here it is against chi-square, both for nearest neighbour, which compares
whole rows.

In [ ]:
# PUZZLE 12 — compress to 100 columns.
# Chapter 5, "Compressing the columns: truncated SVD".
# This cell takes a few minutes.

nn_svd = make_pipeline(TfidfVectorizer(stop_words="english"),
                       TruncatedSVD(n_components=___, random_state=0),
                       KNeighborsClassifier(n_neighbors=1, metric="cosine"))
nn_chosen = make_pipeline(TfidfVectorizer(stop_words="english"),
                          SelectKBest(chi2, k=300),
                          KNeighborsClassifier(n_neighbors=1, metric="cosine"))

svd_scores = cross_val_score(nn_svd, train_all["text"], train_all["cancer_type"],
                             groups=train_all["site"], cv=folds)
chosen_nn_scores = cross_val_score(nn_chosen, train_all["text"], train_all["cancer_type"],
                                   groups=train_all["site"], cv=folds)
print("nearest neighbour, 100 SVD columns:  ", round(svd_scores.mean(), 3))
print("nearest neighbour, 300 chosen words: ", round(chosen_nn_scores.mean(), 3))

assert round(svd_scores.mean(), 3) == 0.535 and round(chosen_nn_scores.mean(), 3) == 0.728

In [ ]:
# PUZZLE 13 — THINK
# Both methods shrink the columns, but chi-square helps nearest neighbour far more than SVD does.
# What does each one look at when it decides what to keep?
# Chapter 5, "Compressing the columns: truncated SVD".
#
# ANSWER: ___

## Open the test pile, once

The setting chosen by cross-validation: TF–IDF with log counts and no stopwords, the 300 words with
the highest chi-square, and logistic regression with `C` = 100. Fit it on all the training and
validation reports, and run it once on the test hospitals.

In [ ]:
# PUZZLE 14 — the pile to fit on: training and validation together.
# Chapter 5, "Opening the test pile".

final = make_pipeline(TfidfVectorizer(sublinear_tf=True, stop_words="english"),
                      SelectKBest(chi2, k=300),
                      LogisticRegression(C=100, max_iter=3000))
final.fit(___["text"], train_all["cancer_type"])
accuracy = (final.predict(test["text"]) == test["cancer_type"]).mean()

assert round(accuracy, 3) == 0.937
print("Test accuracy, this lab:", round(accuracy, 3))
print("Lab 4 (presence, C = 1):", 0.846)
print("Baseline:               ", round(test["cancer_type"].value_counts(normalize=True).max(), 3))

In [ ]:
# PUZZLE 15 — THINK
# Chapter 4 found that the same model scored 0.964 on hospitals it had seen and 0.846 on new
# ones. This lab's model scores 0.937 on new hospitals. Where did most of the gap come from, and
# what in this lab closed it?
# Chapter 5, "Opening the test pile".
#
# ANSWER: ___

## What you did

- Computed idf by hand, and saw that it measures how rare a word is, not what it means.
- Found that one validation pile and five folds of hospitals rank TF–IDF and presence differently,
  and compared settings fold by fold.
- Computed a chi-square by hand, and kept the 300 words most tied to the cancer type: **0.908**
  over five folds, against 0.794 with every word.
- Saw that compressing the columns with SVD helps nearest neighbour much less than choosing words
  by chi-square.
- Tested once: **0.937** on hospitals the model has never seen, against Lab 4's 0.846.